# Phase 2: Baseline Reproduction — SelfCheckGPT

**Goal:** Reproduce the sentence-level hallucination detection results from [Manakul et al., 2023](https://arxiv.org/abs/2303.08896) on the `wiki_bio_gpt3_hallucination` dataset using the `selfcheckgpt` package.

**Methods:**
1. SelfCheck-Unigram (n-gram baseline)
2. SelfCheck-BERTScore
3. SelfCheck-NLI (DeBERTa-v3-large) — best method in the paper

**Runtime:** Use **T4 GPU**. Expected total runtime ~15-25 minutes.

## 0. Setup

In [ ]:
!git clone https://github.com/pragnya-suresh18/LLM-Hallucination-Detection.git
%cd LLM-Hallucination-Detection

In [ ]:
!pip install -q selfcheckgpt datasets bert-score spacy scipy scikit-learn
!python -m spacy download en_core_web_sm -q

In [ ]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Load Dataset

In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from tqdm.notebook import tqdm
from datasets import load_dataset
from sklearn.metrics import average_precision_score
from scipy.stats import pearsonr

warnings.filterwarnings("ignore")

dataset = load_dataset("potsawee/wiki_bio_gpt3_hallucination", split="evaluation")
print(f"Loaded {len(dataset)} passages")
print(f"Features: {list(dataset.features.keys())}")

In [ ]:
label_map = {"accurate": 0, "minor_inaccurate": 1, "major_inaccurate": 1}

passages = []
for idx, ex in enumerate(dataset):
    passages.append({
        "passage_id": idx,
        "passage_text": ex["gpt3_text"],
        "sentences": ex["gpt3_sentences"],
        "labels": [label_map[a] for a in ex["annotation"]],
        "samples": ex["gpt3_text_samples"],
    })

total_sents = sum(len(p["sentences"]) for p in passages)
print(f"{len(passages)} passages, {total_sents} sentences")
print(f"Hallucination rate: {sum(l for p in passages for l in p['labels']) / total_sents:.2%}")

## 2. Evaluation Helpers

In [ ]:
def run_selfcheck_method(method_name, predict_fn, passages):
    """Run a SelfCheck method on all passages, return sentence-level scores."""
    all_scores, all_labels, passage_scores = [], [], []

    for p in tqdm(passages, desc=method_name):
        scores = predict_fn(p)
        all_scores.extend(scores.tolist())
        all_labels.extend(p["labels"])
        passage_scores.append(np.mean(scores))

    return np.array(all_scores), np.array(all_labels), np.array(passage_scores)


def compute_metrics(scores, labels, passage_scores, passages):
    """Compute NonFact AUC-PR, Factual AUC-PR, Passage-level Pearson Corr."""
    nonfact_aucpr = average_precision_score(labels, scores) * 100
    factual_aucpr = average_precision_score(1 - labels, -scores) * 100

    passage_avg_labels = np.array([np.mean(p["labels"]) for p in passages])
    pcc, _ = pearsonr(passage_scores, passage_avg_labels)
    ranking_pcc = pcc * 100

    return {
        "nonfact_aucpr": round(nonfact_aucpr, 2),
        "factual_aucpr": round(factual_aucpr, 2),
        "ranking_pcc": round(ranking_pcc, 2),
    }

## 3a. SelfCheck-Unigram (~1 min)

In [ ]:
from selfcheckgpt.modeling_selfcheck import SelfCheckNgram

selfcheck_ngram = SelfCheckNgram(n=1)

def predict_ngram(p):
    result = selfcheck_ngram.predict(
        sentences=p["sentences"],
        passage=p["passage_text"],
        sampled_passages=p["samples"],
    )
    return np.array(result["sent_level"]["avg_neg_logprob"])

t0 = time.time()
ngram_scores, ngram_labels, ngram_passage_scores = run_selfcheck_method(
    "SelfCheck-Unigram", predict_ngram, passages
)
ngram_time = time.time() - t0
ngram_metrics = compute_metrics(ngram_scores, ngram_labels, ngram_passage_scores, passages)

print(f"\nDone in {ngram_time:.1f}s")
print(f"Results: {ngram_metrics}")

## 3b. SelfCheck-BERTScore (~5-10 min on T4)

In [ ]:
from selfcheckgpt.modeling_selfcheck import SelfCheckBERTScore

selfcheck_bertscore = SelfCheckBERTScore(rescale_with_baseline=True)

def predict_bertscore(p):
    return selfcheck_bertscore.predict(
        sentences=p["sentences"],
        sampled_passages=p["samples"],
    )

t0 = time.time()
bert_scores, bert_labels, bert_passage_scores = run_selfcheck_method(
    "SelfCheck-BERTScore", predict_bertscore, passages
)
bert_time = time.time() - t0
bert_metrics = compute_metrics(bert_scores, bert_labels, bert_passage_scores, passages)

print(f"\nDone in {bert_time:.1f}s")
print(f"Results: {bert_metrics}")

## 3c. SelfCheck-NLI (~10-15 min on T4)

In [ ]:
from selfcheckgpt.modeling_selfcheck import SelfCheckNLI

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

selfcheck_nli = SelfCheckNLI(device=device)

def predict_nli(p):
    return selfcheck_nli.predict(
        sentences=p["sentences"],
        sampled_passages=p["samples"],
    )

t0 = time.time()
nli_scores, nli_labels, nli_passage_scores = run_selfcheck_method(
    "SelfCheck-NLI", predict_nli, passages
)
nli_time = time.time() - t0
nli_metrics = compute_metrics(nli_scores, nli_labels, nli_passage_scores, passages)

print(f"\nDone in {nli_time:.1f}s")
print(f"Results: {nli_metrics}")

## 4. Results Comparison with Paper

In [ ]:
PAPER_RESULTS = {
    "SelfCheck-BERTScore": {"nonfact_aucpr": 81.96, "factual_aucpr": 44.23, "ranking_pcc": 58.18},
    "SelfCheck-Unigram":   {"nonfact_aucpr": 85.63, "factual_aucpr": 58.47, "ranking_pcc": 64.71},
    "SelfCheck-NLI":       {"nonfact_aucpr": 92.50, "factual_aucpr": 66.08, "ranking_pcc": 74.14},
}

our_results = {
    "SelfCheck-Unigram": ngram_metrics,
    "SelfCheck-BERTScore": bert_metrics,
    "SelfCheck-NLI": nli_metrics,
}

rows = []
for method in ["SelfCheck-Unigram", "SelfCheck-BERTScore", "SelfCheck-NLI"]:
    ours = our_results[method]
    paper = PAPER_RESULTS[method]
    rows.append({
        "Method": method,
        "NonFact AUC-PR (Ours)": ours["nonfact_aucpr"],
        "NonFact AUC-PR (Paper)": paper["nonfact_aucpr"],
        "Factual AUC-PR (Ours)": ours["factual_aucpr"],
        "Factual AUC-PR (Paper)": paper["factual_aucpr"],
        "Ranking PCC (Ours)": ours["ranking_pcc"],
        "Ranking PCC (Paper)": paper["ranking_pcc"],
    })

df_results = pd.DataFrame(rows).set_index("Method")
df_results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.1)

Path("data/figures").mkdir(parents=True, exist_ok=True)
Path("data/baseline_results").mkdir(parents=True, exist_ok=True)

methods = ["SelfCheck-Unigram", "SelfCheck-BERTScore", "SelfCheck-NLI"]
metrics_list = ["nonfact_aucpr", "factual_aucpr", "ranking_pcc"]
metric_titles = ["NonFact AUC-PR", "Factual AUC-PR", "Ranking PCC"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric, title in zip(axes, metrics_list, metric_titles):
    ours_vals = [our_results[m][metric] for m in methods]
    paper_vals = [PAPER_RESULTS[m][metric] for m in methods]

    x = np.arange(len(methods))
    width = 0.35

    bars1 = ax.bar(x - width/2, paper_vals, width, label="Paper", color="#5C6BC0", edgecolor="black", linewidth=0.5)
    bars2 = ax.bar(x + width/2, ours_vals, width, label="Ours", color="#EF5350", edgecolor="black", linewidth=0.5)

    ax.set_title(title, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace("SelfCheck-", "") for m in methods], rotation=15)
    ax.legend()
    ax.set_ylim(0, 100)

    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{bar.get_height():.1f}", ha="center", va="bottom", fontsize=9)

plt.suptitle("Baseline Reproduction: Our Results vs Paper (Manakul et al., 2023)", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.savefig("data/figures/baseline_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Save Results & Scores

In [ ]:
Path("data/baseline_results").mkdir(parents=True, exist_ok=True)

np.savez("data/baseline_results/ngram_scores.npz", scores=ngram_scores, labels=ngram_labels)
np.savez("data/baseline_results/bertscore_scores.npz", scores=bert_scores, labels=bert_labels)
np.savez("data/baseline_results/nli_scores.npz", scores=nli_scores, labels=nli_labels)

all_results = {
    "our_results": our_results,
    "paper_results": PAPER_RESULTS,
    "runtimes_seconds": {
        "SelfCheck-Unigram": round(ngram_time, 1),
        "SelfCheck-BERTScore": round(bert_time, 1),
        "SelfCheck-NLI": round(nli_time, 1),
    },
    "dataset_info": {
        "num_passages": len(passages),
        "num_sentences": total_sents,
        "device": str(device),
    },
}

with open("data/baseline_results/baseline_comparison.json", "w") as f:
    json.dump(all_results, f, indent=2)

print(json.dumps(all_results, indent=2))

In [ ]:
print("\nTo push results back to your repo:")
print("  !git add data/ && git commit -m 'Phase 2: baseline reproduction results' && git push")